# Install Libraries


In [11]:
# Install necessary libraries
!pip install --upgrade -q gspread scikit-learn pandas

# Access Data Base

In [12]:
#Google Sheets Linking (Needs user authorization)
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [13]:
import pandas as pd

# Open the spreadsheet by URL
spreadsheet_url = 'https://docs.google.com/spreadsheets/d/1swxApDpgutQxNKr6GJHSr96PQZUxd_qcKAx8hlV7YOI/edit?usp=drive_link'
wb = gc.open_by_url(spreadsheet_url)
sheet = wb.get_worksheet(0)

# Load data into DataFrame
data = sheet.get_all_values()
df = pd.DataFrame(data[1:], columns=data[0])

print("Sanity Check - Original Data:")
display(df.head())

Sanity Check - Original Data:


,text,label_specific,label_generic
0,"Hello Michael, your account needs immediate at...",llm_phishing,phishing
1,"Dear Sarah, I trust this email finds you well....",llm_legit,legit
2,"Dear Ms. Jessica Davis, I trust this email fin...",llm_legit,legit
3,----------------------------------------------...,human_legit,legit
4,"Update account activity Hello jose@monkey.org,...",human_phishing,phishing


# Vectorization

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorization (N-grams + TF-IDF)
# Using unigrams and bigrams (1, 2)
print("Vectorizing text...")
tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=10000)
X = tfidf.fit_transform(df['text'].astype(str))

Vectorizing text...


## Split Data into Training and Evaluation Sets

Split the original `df` (which contains raw text and labels) into training (`df_train`) and evaluation (`df_eval`) sets.


In [16]:
from sklearn.model_selection import train_test_split

# Create the generic label column in the main DataFrame first
label_mapping = {
    'llm_legit': 0,
    'human_legit': 0,
    'llm_phishing': 1,
    'human_phishing': 1
}
df['label_generic'] = df['label_specific'].map(label_mapping)

# Split the indices of the DataFrame into training and evaluation sets
# Stratify by 'label_generic' to maintain class distribution
train_indices, eval_indices = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=df['label_generic']
)

# Use the split indices to create df_train and df_eval
df_train = df.loc[train_indices].copy()
df_eval = df.loc[eval_indices].copy()

print("Shape of df_train:", df_train.shape)
print("Shape of df_eval:", df_eval.shape)

print("\nSanity Check - df_train head:")
display(df_train.head())

print("\nSanity Check - df_eval head:")
display(df_eval.head())

Shape of df_train: (2560, 3)
Shape of df_eval: (640, 3)

Sanity Check - df_train head:


,text,label_specific,label_generic
524,WeTransfer jose@monkey.orgYou have received ...,human_phishing,1
1218,"On Fri, Feb 08, 2008 at 03:00:34PM -0500, Feli...",human_legit,0
2460,Account Storage *jose@monkey.org* Password has...,human_phishing,1
2386,"Dear Michael, I hope this email finds you in g...",llm_phishing,1
1798,"Dear Sarah, I hope this email finds you well. ...",llm_legit,0



Sanity Check - df_eval head:


,text,label_specific,label_generic
1841,"From: ""Gene Heskett"" \nSent: Friday, 2008, Feb...",human_legit,0
2009,"Dear Matthew, We hope you're doing well. We're...",llm_legit,0
3065,Verify your MetaMask Wallet Our system has sho...,human_phishing,1
2118,http://issues.apache.org/SpamAssassin/show_bug...,human_legit,0
2161,"Dear Michael, We have detected some suspicious...",llm_phishing,1


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TfidfVectorizer with specified parameters
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=10000)

# Fit the vectorizer on the 'text' column of the training data and transform it
X_train = tfidf_vectorizer.fit_transform(df_train['text'].astype(str))

print("Shape of X_train:", X_train.shape)
print("First 5 rows of X_train (sparse matrix preview):")
print(X_train[:5].toarray())

Shape of X_train: (2560, 10000)
First 5 rows of X_train (sparse matrix preview):
[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.08255817 0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]


## Encode Generic Labels for Training

Create the numerical target variable `y_train_generic` for 'legit' vs. 'phishing' classification from `df_train`.


In [18]:
label_mapping = {
    'llm_legit': 0,
    'human_legit': 0,
    'llm_phishing': 1,
    'human_phishing': 1
}
df_train['label_generic'] = df_train['label_specific'].map(label_mapping)
y_train_generic = df_train['label_generic']

print("Shape of y_train_generic:", y_train_generic.shape)
print("Value counts for y_train_generic:")
print(y_train_generic.value_counts())
print("\nSanity Check - df_train head with new 'label_generic' column:")
display(df_train.head())

Shape of y_train_generic: (2560,)
Value counts for y_train_generic:
label_generic
1    1280
0    1280
Name: count, dtype: int64

Sanity Check - df_train head with new 'label_generic' column:


,text,label_specific,label_generic
524,WeTransfer jose@monkey.orgYou have received ...,human_phishing,1
1218,"On Fri, Feb 08, 2008 at 03:00:34PM -0500, Feli...",human_legit,0
2460,Account Storage *jose@monkey.org* Password has...,human_phishing,1
2386,"Dear Michael, I hope this email finds you in g...",llm_phishing,1
1798,"Dear Sarah, I hope this email finds you well. ...",llm_legit,0


## Build Base Model (Legit vs. Phishing)

Train a `LinearSVC` model (the base model) using the vectorized training data (`X_train`) and `y_train_generic`.


In [20]:
from sklearn.svm import LinearSVC

# Instantiate LinearSVC model
# Using random_state for reproducibility
svm_generic = LinearSVC(random_state=42, dual=False)

# Train the model
print("Training LinearSVC model...")
svm_generic.fit(X_train, y_train_generic)

print("LinearSVC model trained successfully.")

Training LinearSVC model...
LinearSVC model trained successfully.


## Preprocess Evaluation Data

Use the same fitted `TfidfVectorizer` from the training step to transform the `text` column of `df_eval` into `X_eval`.


In [21]:
X_eval = tfidf_vectorizer.transform(df_eval['text'].astype(str))

print("Shape of X_eval:", X_eval.shape)

Shape of X_eval: (640, 10000)


## Encode Eval/Test Set Labels

Create the numerical target variables for the evaluation set: `y_eval_generic` (legit/phishing), and derived specific labels (`y_eval_legit_specific` for human_legit/llm_legit and `y_eval_phishing_specific` for human_phishing/llm_phishing) from `df_eval`.


In [22]:
label_mapping = {
    'llm_legit': 0,
    'human_legit': 0,
    'llm_phishing': 1,
    'human_phishing': 1
}
df_eval['label_generic'] = df_eval['label_specific'].map(label_mapping)
y_eval_generic = df_eval['label_generic']

# Filter for legit emails and create specific labels
df_eval_legit_specific = df_eval[df_eval['label_generic'] == 0].copy()
legit_specific_mapping = {
    'human_legit': 0,
    'llm_legit': 1
}
y_eval_legit_specific = df_eval_legit_specific['label_specific'].map(legit_specific_mapping)

# Filter for phishing emails and create specific labels
df_eval_phishing_specific = df_eval[df_eval['label_generic'] == 1].copy()
phishing_specific_mapping = {
    'human_phishing': 0,
    'llm_phishing': 1
}
y_eval_phishing_specific = df_eval_phishing_specific['label_specific'].map(phishing_specific_mapping)


print("Shape of y_eval_generic:", y_eval_generic.shape)
print("Value counts for y_eval_generic:")
print(y_eval_generic.value_counts())

print("\nShape of y_eval_legit_specific:", y_eval_legit_specific.shape)
print("Value counts for y_eval_legit_specific:")
print(y_eval_legit_specific.value_counts())

print("\nShape of y_eval_phishing_specific:", y_eval_phishing_specific.shape)
print("Value counts for y_eval_phishing_specific:")
print(y_eval_phishing_specific.value_counts())

print("\nSanity Check - df_eval head with new 'label_generic' column:")
display(df_eval.head())

Shape of y_eval_generic: (640,)
Value counts for y_eval_generic:
label_generic
0    320
1    320
Name: count, dtype: int64

Shape of y_eval_legit_specific: (320,)
Value counts for y_eval_legit_specific:
label_specific
1    161
0    159
Name: count, dtype: int64

Shape of y_eval_phishing_specific: (320,)
Value counts for y_eval_phishing_specific:
label_specific
1    162
0    158
Name: count, dtype: int64

Sanity Check - df_eval head with new 'label_generic' column:


,text,label_specific,label_generic
1841,"From: ""Gene Heskett"" \nSent: Friday, 2008, Feb...",human_legit,0
2009,"Dear Matthew, We hope you're doing well. We're...",llm_legit,0
3065,Verify your MetaMask Wallet Our system has sho...,human_phishing,1
2118,http://issues.apache.org/SpamAssassin/show_bug...,human_legit,0
2161,"Dear Michael, We have detected some suspicious...",llm_phishing,1


## Define Custom Cost Function

Implement a Python function to calculate the custom cost: Cost = (15 * False Negatives) + False Positives.


In [23]:
from sklearn.metrics import confusion_matrix

def calculate_custom_cost(y_true, y_pred):
    """
    Calculates a custom cost based on False Negatives and False Positives.

    Args:
        y_true (array-like): True labels.
        y_pred (array-like): Predicted labels.

    Returns:
        float: The calculated custom cost.
    """
    # Compute the confusion matrix. For binary classification with labels [0, 1]:
    # [[TN, FP],
    #  [FN, TP]]
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    FP = cm[0, 1] # False Positives: Actual 0, Predicted 1
    FN = cm[1, 0] # False Negatives: Actual 1, Predicted 0

    cost = (15 * FN) + FP
    return cost

print("Custom cost function 'calculate_custom_cost' defined.")

Custom cost function 'calculate_custom_cost' defined.


## Get Predictions for Base Model

Generate predictions for the `svm_generic` model on the vectorized evaluation data (`X_eval`).


In [24]:
y_pred_generic = svm_generic.predict(X_eval)

print("Shape of y_pred_generic:", y_pred_generic.shape)
print("First 5 predictions of y_pred_generic:")
print(y_pred_generic[:5])

Shape of y_pred_generic: (640,)
First 5 predictions of y_pred_generic:
[0 0 1 0 1]


## Evaluate Base Model on Generic Labels

Calculate and report standard metrics (accuracy, classification report, confusion matrix) and the custom cost using `y_eval_generic` and the predictions from `svm_generic`.


In [25]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Calculate accuracy
accuracy = accuracy_score(y_eval_generic, y_pred_generic)

# Generate classification report
report = classification_report(y_eval_generic, y_pred_generic, target_names=['legit', 'phishing'])

# Compute confusion matrix
cm = confusion_matrix(y_eval_generic, y_pred_generic)

# Calculate custom cost
custom_cost = calculate_custom_cost(y_eval_generic, y_pred_generic)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)
print("\nConfusion Matrix:")
print(cm)
print(f"\nCustom Cost (15*FN + FP): {custom_cost}")

Accuracy: 0.9875

Classification Report:
              precision    recall  f1-score   support

       legit       0.98      1.00      0.99       320
    phishing       1.00      0.98      0.99       320

    accuracy                           0.99       640
   macro avg       0.99      0.99      0.99       640
weighted avg       0.99      0.99      0.99       640


Confusion Matrix:
[[319   1]
 [  7 313]]

Custom Cost (15*FN + FP): 106


## Analyze Base Model's Performance on Specific Subsets

Break down the performance of the `svm_generic` model across the specific sub-categories (human_legit, llm_legit, human_phishing, llm_phishing) within the evaluation set.


In [26]:
eval_performance = df_eval.copy()
eval_performance['predicted_generic'] = y_pred_generic

print("Shape of eval_performance:", eval_performance.shape)
print("Sanity Check - eval_performance head with new 'predicted_generic' column:")
display(eval_performance.head())

Shape of eval_performance: (640, 4)
Sanity Check - eval_performance head with new 'predicted_generic' column:


,text,label_specific,label_generic,predicted_generic
1841,"From: ""Gene Heskett"" \nSent: Friday, 2008, Feb...",human_legit,0,0
2009,"Dear Matthew, We hope you're doing well. We're...",llm_legit,0,0
3065,Verify your MetaMask Wallet Our system has sho...,human_phishing,1,1
2118,http://issues.apache.org/SpamAssassin/show_bug...,human_legit,0,0
2161,"Dear Michael, We have detected some suspicious...",llm_phishing,1,1


In [27]:
print("Predicted generic labels grouped by specific labels:")
display(eval_performance.groupby('label_specific')['predicted_generic'].value_counts().unstack(fill_value=0))

Predicted generic labels grouped by specific labels:


predicted_generic,0,1
label_specific,,
human_legit,159,0
human_phishing,2,156
llm_legit,160,1
llm_phishing,5,157


## Summary:

### Data Analysis Key Findings

*   The initial dataset (`df`) was successfully split into training (`df_train`) and evaluation (`df_eval`) sets with 2560 and 640 entries, respectively.
*   A `TfidfVectorizer` was fitted on the training data, resulting in a feature matrix `X_train` of shape (2560, 10000) and `X_eval` of shape (640, 10000).
*   Generic labels for 'legit' (0) vs. 'phishing' (1) were created for both training (`y_train_generic`) and evaluation (`y_eval_generic`) sets. Both sets exhibited a balanced distribution of 1280 legit/1280 phishing instances for training and 320 legit/320 phishing instances for evaluation.
*   Specific sub-category labels were also derived for the evaluation set: `y_eval_legit_specific` (159 human_legit, 161 llm_legit) and `y_eval_phishing_specific` (158 human_phishing, 162 llm_phishing).
*   A `LinearSVC` model was trained to classify generic 'legit' vs. 'phishing' emails.
*   The custom cost function was defined as (15 \* False Negatives) + False Positives.
*   The generic model achieved an overall accuracy of **0.9875** on the evaluation set.
*   The classification report shows strong performance:
    *   For 'legit' (class 0): Precision of 0.98, Recall of 1.00, F1-score of 0.99.
    *   For 'phishing' (class 1): Precision of 1.00, Recall of 0.98, F1-score of 0.99.
*   The confusion matrix indicated:
    *   319 true negatives (legit emails correctly identified as legit).
    *   1 false positive (legit email incorrectly classified as phishing).
    *   7 false negatives (phishing emails incorrectly classified as legit).
    *   313 true positives (phishing emails correctly identified as phishing).
*   The calculated custom cost for the generic model was **106**.
*   Performance breakdown by specific sub-categories revealed:
    *   All 159 `human_legit` emails were correctly classified.
    *   160 out of 161 `llm_legit` emails were correctly classified, with 1 being misclassified as phishing.
    *   156 out of 158 `human_phishing` emails were correctly classified, with 2 being misclassified as legit.
    *   157 out of 162 `llm_phishing` emails were correctly classified, with 5 being misclassified as legit.

### Insights or Next Steps

*   The model exhibits high overall performance, but the custom cost highlights the severe penalty for misclassifying phishing emails as legitimate. The 7 false negatives, predominantly from LLM-generated phishing emails (5 out of 7), contribute significantly to this cost.
*   Further investigation is warranted into the characteristics of the 5 LLM-generated phishing emails that were misclassified as legitimate. This could involve exploring additional features or fine-tuning the model to specifically reduce false negatives for LLM-generated content, potentially by adjusting the classification threshold.
